In [0]:
import delta
import sys

sys.path.insert(0, "../lib/")

import utils
import ingestors

In [0]:
catalog = "bronze"
schemaname = "upsell"

## usar como exemplo
# tablename = "transacao_produto"
# id_field = "IdTransacao"
# timestamp_field = "_extracted_at"

tablename = dbutils.widgets.get("tablename")
id_field = dbutils.widgets.get("id_field")
timestamp_field = dbutils.widgets.get("timestamp_field")

full_load_path = f"/Volumes/raw/{schemaname}/full_load/{tablename}/"
cdc_path = f"/Volumes/raw/{schemaname}/cdc/{tablename}/"
checkpoint_location = f"/Volumes/raw/{schemaname}/cdc/{tablename}/_checkpoints/"

In [0]:
if not utils.table_exists(spark, catalog, schemaname, tablename):

        print("Tabela não existente, criando...")

        dbutils.fs.rm(checkpoint_location, True)

        ingest_full_load = ingestors.Ingestor(spark=spark,
                                              catalog=catalog,
                                              schemaname=schemaname,
                                              tablename=tablename,
                                              data_format="parquet")
        
        ingest_full_load.execute(full_load_path)

        print("Tabela criada com sucesso!")
        
else:
        print("Tabela existente, ignorando full-load")

In [0]:
ingest_cdc = ingestors.IngestorCDC(spark=spark, catalog=catalog, schemaname=schemaname, tablename=tablename, data_format="parquet", id_field=id_field, timestamp_field=timestamp_field)


stream = ingest_cdc.execute(cdc_path)